# 신용카드 채무불이행 고객 예측 -

https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset

In [ ]:
#!pip install xlrd --break-system-packages


In [3]:
import pandas as pd
import urllib.request
import os

os.makedirs('./data', exist_ok=True)

# UCI 원본
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"
urllib.request.urlretrieve(url, './data/UCI_Credit_Card.xls')

# xls 읽기
df = pd.read_excel('./data/UCI_Credit_Card.xls', header=1)
df.to_csv('./data/UCI_Credit_Card.csv', index=False)
print(df.shape)

# df = pd.read_csv('./data/UCI_Credit_Card.csv')
card_df = df.drop('ID', axis=1)
card_df.head(3)

(30000, 25)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0


In [4]:
card_df = card_df.rename(columns={'default payment next month': 'default'})

y_target = card_df['default']
X_features = card_df.drop('default', axis=1)


# Default 예측 모델링 프로세스 (Recall Focus)
## 1) 목표/평가지표 확정
비즈니스 목표: 부실 가능 고객을 놓치지 않기
1순위 지표: Recall
보조 지표: Precision, F1, PR-AUC
운영 기준 예시:
Recall 최소 기준(예: 0.80 또는 0.85) 먼저 만족
그 조건에서 Precision 최대화

# 기초 EDA (분할 후 Train 기준)
타깃 불균형 정도 확인

결측치 비율/패턴 확인

변수 타입 점검 (수치형/범주형)

이상치 후보 확인

목적:

전처리 전략 근거 확보

Recall 저해 요인 사전 파악

In [5]:
# 1단계: 기본 현황 파악
print("=== Shape ===")
print(card_df.shape)

print("\n=== 데이터 타입 ===")
print(card_df.dtypes)

print("\n=== 결측치 확인 ===")
print(card_df.isnull().sum())

=== Shape ===
(30000, 24)

=== 데이터 타입 ===
LIMIT_BAL    int64
SEX          int64
EDUCATION    int64
MARRIAGE     int64
AGE          int64
PAY_0        int64
PAY_2        int64
PAY_3        int64
PAY_4        int64
PAY_5        int64
PAY_6        int64
BILL_AMT1    int64
BILL_AMT2    int64
BILL_AMT3    int64
BILL_AMT4    int64
BILL_AMT5    int64
BILL_AMT6    int64
PAY_AMT1     int64
PAY_AMT2     int64
PAY_AMT3     int64
PAY_AMT4     int64
PAY_AMT5     int64
PAY_AMT6     int64
default      int64
dtype: object

=== 결측치 확인 ===
LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_0        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
default      0
dtype: int64


In [8]:
# 기초 통계량
card_df.describe()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,167484.322667,1.603733,1.853133,1.551867,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,-0.266200,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,129747.661567,0.489129,0.790349,0.521970,9.217904,1.123802,1.197186,1.196868,1.169139,1.133187,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,10000.000000,1.000000,0.000000,0.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,1000000.000000,2.000000,6.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


1단계 결과 요약

✅ 결측치 없음 — 모든 컬럼 30,000개 완전
✅ 모든 컬럼 int64 — 별도 타입 변환 불필요

주목할 이상값들:

EDUCATION: min=0, max=6 → 공식 코드북엔 1~4만 정의 (0, 5, 6은 이상값)
MARRIAGE: min=0, max=3 → 공식 코드북엔 1~3만 정의 (0은 이상값)
BILL_AMT 시리즈: 음수값 존재 (예: min=-170,000) → 과납/환급 케이스
PAY_AMT 시리즈: max가 매우 큼 (1.6M) → 극단적 이상값 가능성

In [9]:
# 2단계: 타겟(default) 분포 확인
print("=== Default 클래스 분포 ===")
print(card_df['default'].value_counts())
print()
print("=== 비율 ===")
print(card_df['default'].value_counts(normalize=True).round(3))

=== Default 클래스 분포 ===
default
0    23364
1     6636
Name: count, dtype: int64

=== 비율 ===
default
0    0.779
1    0.221
Name: proportion, dtype: float64


2단계 결과 요약

정상(0): 23,364명 (77.9%)
채무불이행(1): 6,636명 (22.1%)
약 3.5:1 불균형 → 나중에 전처리 시 대응 필요 (SMOTE 또는 class_weight 등)

In [10]:
# 3단계: 범주형 피처 값 분포
print("=== EDUCATION 값 분포 ===")
print(card_df['EDUCATION'].value_counts().sort_index())

print("\n=== MARRIAGE 값 분포 ===")
print(card_df['MARRIAGE'].value_counts().sort_index())

print("\n=== SEX 값 분포 ===")
print(card_df['SEX'].value_counts().sort_index())

=== EDUCATION 값 분포 ===
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

=== MARRIAGE 값 분포 ===
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64

=== SEX 값 분포 ===
SEX
1    11888
2    18112
Name: count, dtype: int64


In [11]:
# PAY_0 ~ PAY_6 고유값 확인
pay_cols = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']
for col in pay_cols:
    print(f"{col}: {sorted(card_df[col].unique())}")

PAY_0: [np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
PAY_2: [np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
PAY_3: [np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
PAY_4: [np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
PAY_5: [np.int64(-2), np.int64(-1), np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
PAY_6: [np.int64(-2), np.int64(-1), np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


3단계 결과 요약범주형 이상값:

EDUCATION: 0(14개), 5(280개), 6(51개) → 총 345개 이상값 (코드북 정의 외)
MARRIAGE: 0(54개) → 이상값
SEX: 1/2만 존재 → 정상
PAY 시리즈:

-2, -1, 0 모두 존재 → 정상 (각각 잔액없음/정상결제/최소결제 의미)
1~8: 연체 개월 수 → 정상

In [12]:
# 4단계: 피처별 Default 비율
cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE']
for col in cat_cols:
    print(f"=== {col}별 Default 비율 ===")
    print(card_df.groupby(col)['default'].mean().round(3))
    print()

=== SEX별 Default 비율 ===
SEX
1    0.242
2    0.208
Name: default, dtype: float64

=== EDUCATION별 Default 비율 ===
EDUCATION
0    0.000
1    0.192
2    0.237
3    0.252
4    0.057
5    0.064
6    0.157
Name: default, dtype: float64

=== MARRIAGE별 Default 비율 ===
MARRIAGE
0    0.093
1    0.235
2    0.209
3    0.260
Name: default, dtype: float64



In [13]:
pay_cols = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']
for col in pay_cols:
    print(f"=== {col}별 Default 비율 ===")
    print(card_df.groupby(col)['default'].mean().round(3))
    print()

=== PAY_0별 Default 비율 ===
PAY_0
-2    0.132
-1    0.168
 0    0.128
 1    0.339
 2    0.691
 3    0.758
 4    0.684
 5    0.500
 6    0.545
 7    0.778
 8    0.579
Name: default, dtype: float64

=== PAY_2별 Default 비율 ===
PAY_2
-2    0.183
-1    0.160
 0    0.159
 1    0.179
 2    0.556
 3    0.617
 4    0.505
 5    0.600
 6    0.750
 7    0.600
 8    0.000
Name: default, dtype: float64

=== PAY_3별 Default 비율 ===
PAY_3
-2    0.185
-1    0.156
 0    0.175
 1    0.250
 2    0.516
 3    0.575
 4    0.579
 5    0.571
 6    0.609
 7    0.815
 8    0.667
Name: default, dtype: float64

=== PAY_4별 Default 비율 ===
PAY_4
-2    0.193
-1    0.159
 0    0.183
 1    0.500
 2    0.523
 3    0.611
 4    0.667
 5    0.514
 6    0.400
 7    0.828
 8    0.500
Name: default, dtype: float64

=== PAY_5별 Default 비율 ===
PAY_5
-2    0.197
-1    0.162
 0    0.189
 2    0.542
 3    0.635
 4    0.607
 5    0.588
 6    0.750
 7    0.828
 8    1.000
Name: default, dtype: float64

=== PAY_6별 Default 비율 ===
PAY_6
-2   

In [14]:
# 수치형 피처와 default 상관관계
corr = card_df.corr()['default'].drop('default').sort_values(ascending=False)
print("=== default와 상관관계 (상위/하위) ===")
print(corr)

=== default와 상관관계 (상위/하위) ===
PAY_0        0.324794
PAY_2        0.263551
PAY_3        0.235253
PAY_4        0.216614
PAY_5        0.204149
PAY_6        0.186866
EDUCATION    0.028006
AGE          0.013890
BILL_AMT6   -0.005372
BILL_AMT5   -0.006760
BILL_AMT4   -0.010156
BILL_AMT3   -0.014076
BILL_AMT2   -0.014193
BILL_AMT1   -0.019644
MARRIAGE    -0.024339
SEX         -0.039961
PAY_AMT6    -0.053183
PAY_AMT5    -0.055124
PAY_AMT3    -0.056250
PAY_AMT4    -0.056827
PAY_AMT2    -0.058579
PAY_AMT1    -0.072929
LIMIT_BAL   -0.153520
Name: default, dtype: float64


4단계 결과 요약
범주형 피처:

SEX: 남성(1) 24.2% > 여성(2) 20.8% → 차이 미미
EDUCATION: 이상값(0,4,5,6)은 샘플 수 적고 비율도 튀어있음 → 전처리 필요
MARRIAGE: 이상값(0)은 9.3%로 낮음 → 전처리 필요

PAY 시리즈 (핵심 발견!):

연체 1개월(1)만 돼도 Default 비율 급등 (PAY_0 기준: 0→13% → 1→34% → 2→69%)
PAY_0이 가장 강력한 예측 피처 (상관관계 0.325로 1위)

상관관계 전체:

상위: PAY_0~PAY_6 (연체 관련) → 예측력 높음
하위: BILL_AMT 시리즈 → 상관관계 거의 없음
LIMIT_BAL (-0.154) → 한도 높을수록 Default 낮음 (합리적)

EDA 종합 결론 → 전처리 시 할 일들:

EDUCATION 이상값(0,5,6) → 처리 방안 필요
MARRIAGE 이상값(0) → 처리 방안 필요
클래스 불균형 (78:22) → 모델링 시 대응 필요
BILL_AMT 음수값 → 처리 방안 검토